# Treinamento

As colunas da database são: PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked

### Pré processamneto realizado
    As colunas 'PassengerId',"Name",'Ticket',"Cabin" foram cortadas por serem consideradas ruído ou pouco significativas para o resultado.
    A coluna 'Sex' foi transformada em valor binário
    A coluna 'Embarked' foi transformada em 3 colunas binárias cada uma referente a um dos valores possiveis de onde a pessoa embarcou.

### Modelo usado
    O xgboost foi usado como modelo caixa preta que cria uma série de arvores de decisão (Tradicionalmente considerados caixa branca) e usa o resultado dessas arvores para tomar uma decisão, de modo que a sáida é matematicamente complexa o suficiente para ser humamente dificil explicar.

### Hiperparametros
    A taxa de aprendizado é de 0.1 e a profundidade da árvore é de 5. Outras combinações de valores foram tentadas mas essa atingiu o maior resultado em termos de acurácia (0.8182).
    _Obs_: Usando os parametros padrão do xgboost a acurácia foi de 0.7622

### Valores alterados
    Houveram tentativas de ajuste menores, como cortar outras colunas, mas todas as combinações e variações diferentes das que estão na célula abaixo apresentaram redução na acurácia.
    

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

dataset = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(dataset)

df = df.drop(['PassengerId',"Name",'Ticket',"Cabin"],axis=1).dropna()

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df = pd.get_dummies(df, columns=['Embarked'])

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(learning_rate=0.1,max_depth=5)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Acurácia: {accuracy:.4f}")

Acurácia: 0.8182


### Seleciona um passageiro e pega os valores de Shap dele.
São criadas 5 grupos de valores diferentes, cada um deles gerado usando um numero de exemplos diferentes. (Pode ser vareiado modificando valores na lista nsamples_lista alterando n na chamada na função explainer.shap_values, ex: nsamples=n*5). 

In [2]:
import shap
import numpy as np

indice_passageiro = 0

instancia = X_test.iloc[indice_passageiro:indice_passageiro+1]

print(f"--- Dados do Passageiro (Índice {indice_passageiro}) ---")
for coluna, valor in instancia.iloc[0].items():
    print(f"  {coluna}: {valor}")
print("-" * 50 + "\n")

background = shap.sample(X_train, 10)

def predict_fn(X):
    return model.predict_proba(X)

explainer = shap.KernelExplainer(predict_fn, background)

nsamples_lista = [10, 15, 20, 25, 30]
explicacoes_pobres = []

for n in nsamples_lista:
    shap_values = explainer.shap_values(instancia, nsamples=n*5, silent=True)
    valores_brutos = np.array(shap_values)
    if valores_brutos.ndim == 3:
        valores = valores_brutos[0, :, 1]
    elif isinstance(shap_values, list):
        valores = shap_values[1][0]
    else:
        valores = valores_brutos[0]
        
    exp_formatada = [(X_train.columns[i], float(valores[i])) for i in range(len(X_train.columns))]
    explicacoes_pobres.append(exp_formatada)

for idx, exp in enumerate(explicacoes_pobres):
    print(f"--- Explicação {idx+1} (nsamples={nsamples_lista[idx]}) ---")
    exp_ordenada = sorted(exp, key=lambda x: abs(x[1]), reverse=True)
    for feature, valor in exp_ordenada:
        print(f"  {feature}: {valor:.4f}")
    print()

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Dados do Passageiro (Índice 0) ---
  Pclass: 1
  Sex: 0
  Age: 24.0
  SibSp: 0
  Parch: 0
  Fare: 69.3
  Embarked_C: True
  Embarked_Q: False
  Embarked_S: False
--------------------------------------------------

--- Explicação 1 (nsamples=10) ---
  Sex: 0.3488
  Pclass: 0.0833
  Age: -0.0462
  Fare: 0.0193
  Parch: -0.0165
  SibSp: 0.0089
  Embarked_C: 0.0049
  Embarked_S: 0.0000
  Embarked_Q: 0.0000

--- Explicação 2 (nsamples=15) ---
  Sex: 0.3444
  Pclass: 0.0776
  Age: -0.0431
  Fare: 0.0258
  Parch: -0.0194
  Embarked_S: 0.0126
  SibSp: 0.0034
  Embarked_C: 0.0011
  Embarked_Q: 0.0000

--- Explicação 3 (nsamples=20) ---
  Sex: 0.3398
  Pclass: 0.0765
  Age: -0.0422
  Fare: 0.0257
  Parch: -0.0141
  Embarked_S: 0.0111
  Embarked_C: 0.0097
  SibSp: -0.0039
  Embarked_Q: 0.0000

--- Explicação 4 (nsamples=25) ---
  Sex: 0.3397
  Pclass: 0.0744
  Age: -0.0431
  Fare: 0.0290
  Parch: -0.0129
  Embarked_C: 0.0096
  Embarked_S: 0.0080
  SibSp: -0.0022
  Embarked_Q: 0.0000

--- Expl

In [3]:
import os
import requests

_QWEN_WORKING_URL = None

def get_qwen_url():
    global _QWEN_WORKING_URL
    
    if _QWEN_WORKING_URL:
        return _QWEN_WORKING_URL
        
    print("Buscando a URL do Qwen...")
    url_env = os.getenv("QWEN_URL")
    urls_para_testar = [url_env] if url_env else []
    
    urls_para_testar.extend([
        "http://qwen_slm:11434/api/generate",
        "http://qwen:11434/api/generate",
        "http://host.docker.internal:11434/api/generate",
        "http://localhost:11434/api/generate",
        "http://127.0.0.1:11434/api/generate"
    ])
    
    teste_payload = {"model": "qwen2.5", "prompt": "teste", "stream": False}
    
    for url in urls_para_testar:
        try:
            resposta = requests.post(url, json=teste_payload, timeout=30)
            if resposta.status_code == 200:
                _QWEN_WORKING_URL = url
                return _QWEN_WORKING_URL
        except requests.exceptions.RequestException:
            continue
            
    raise ConnectionError("Erro Crítico: Não encontrei o modelo Qwen em nenhuma das rotas.")

def ask_qwen(prompt):
    url = get_qwen_url()
    
    payload = {
        "model": "qwen2.5", 
        "prompt": prompt,
        "stream": False
    }
    
    try:
        resposta = requests.post(url, json=payload, timeout=30)
        if resposta.status_code == 200:
            return resposta.json().get("response", "").strip()
        else:
            return f"Erro na API: Status {resposta.status_code}"
    except requests.exceptions.RequestException as e:
        return f"Erro de conexão: {e}"

In [4]:
valores_reais = instancia.to_dict(orient='records')[0]
textos_pobres = []

for idx, exp in enumerate(explicacoes_pobres):
    detalhes = []
    for feature, shap_val in exp:
        valor_real = valores_reais.get(feature, "N/A")
        detalhes.append(f"- {feature} (Valor real: {valor_real}): Peso SHAP = {shap_val:.4f}")
    
    texto_detalhes = "\n".join(detalhes)
    
    prompt = f"""Você é um especialista em análise de dados. Estou analisando a sobrevivência de UM passageiro do Titanic.

GABARITO DAS VARIÁVEIS:
- Pclass: 1 = 1ª Classe (Alta), 2 = 2ª Classe (Média), 3 = 3ª Classe (Baixa).
- Sex: 0 = Mulher, 1 = Homem.
- Age: Idade em anos.
- SibSp: Número de irmãos ou cônjuges a bordo.
- Parch: Número de pais ou filhos a bordo.
- Fare: Preço pago pela passagem.
- Embarked_C, Embarked_Q, Embarked_S: Valor 1.0 indica o porto em que embarcou.

REGRA DO SHAP (Importância):
- Pesos SHAP POSITIVOS (> 0) significam que o fator AJUDOU a sobreviver.
- Pesos SHAP NEGATIVOS (< 0) significam que o fator CONTRIBUIU PARA A MORTE.
- Quanto maior o valor numérico (ignorando o sinal), maior a influência daquele fator.

Aqui estão as características exatas deste passageiro e os pesos calculados:
{texto_detalhes}

Com base ESTRITAMENTE no gabarito e nestes dados, escreva um parágrafo curto explicando o que mais influenciou a predição para este passageiro. Não invente informações.
Você deve começar dizendo se o passageiro sobreviveu ou morreu e explicar tudo em linguagem natural, sem usar valores nem o nome das váriaveis"""

    resposta_qwen = ask_qwen(prompt)
    textos_pobres.append(resposta_qwen)
    
    print(f"--- Explicação Textual {idx+1} ---")
    print(resposta_qwen)
    print()

Buscando a URL do Qwen...
--- Explicação Textual 1 ---
Baseado no gabarito fornecido e nas características do passageiro, podemos inferir que ele sobreviveu. A predição para a sobrevivência deste passageiro foi influenciada principalmente pela variável "Sex", onde ser mulher foi um fator positivo significativo, contribuindo positivamente para a predição de sobrevivência. Além disso, o fato de ser de primeira classe também teve um pequeno impacto positivo. Por outro lado, a idade do passageiro, embora tenha um peso negativo, teve um efeito menor, dado que ele tem 24 anos, uma idade relativamente jovem e propensa a sobreviver. A ausência de irmãos ou cônjuges a bordo e a baixa passagem (considerando a tarifa paga) também contribuíram, embora de forma muito pequena, para a predição de sobrevivência.

--- Explicação Textual 2 ---
O passageiro sobreviveu. A principal razão para esta predição é o fato de ser uma mulher (Sex = 0), o que teve um peso SHAP bastante positivo, indicando que esse 

In [5]:
textos_combinados = "\n\n".join([f"Explicação {i+1}:\n{texto}" for i, texto in enumerate(textos_pobres)])

prompt_sintese = f"""Você é um especialista em análise de dados avaliando múltiplas interpretações sobre a sobrevivência de UM passageiro do Titanic.

Abaixo estão 5 explicações geradas com base em cálculos matemáticos que contêm ruído proposital. Elas concordam nos pontos principais, mas se contradizem nos detalhes menores.

{textos_combinados}

Sua tarefa:
Analise as 5 explicações acima.
Identifique os fatores de consenso (os mais importantes e consistentes que aparecem em todas).
Descarte as contradições ou os fatores descritos como tendo impacto mínimo.
Escreva um único parágrafo final, definitivo e coerente, em linguagem natural, explicando por que este passageiro sobreviveu ou não. Não liste pontos, faça um texto fluido.
"""

print("Iniciando a síntese com o Qwen local...\n")
explicacao_forte_qwen = ask_qwen(prompt_sintese)

print("=== EXPLICAÇÃO FORTE (QWEN SLM) ===")
print(explicacao_forte_qwen)

Iniciando a síntese com o Qwen local...

=== EXPLICAÇÃO FORTE (QWEN SLM) ===
Este passageiro sobreviveu, principalmente devido a duas principais influências: o fato de ser uma mulher e a alta classe social. A análise SHAP evidencia que ser mulher contribuiu significativamente para a predição positiva, refletindo a prioridade dada a elas durante o resgate. A classe social de 1ª classe também teve um impacto positivo, embora menor, possivelmente devido à prioridade concedida a esta categoria durante as operações de resgate. As outras variáveis, como a idade (24 anos) e a ausência de parentes a bordo, tiveram impactos negativos, embora menores, pois a idade mais avançada e a falta de parentes que poderiam ter aumentado a probabilidade de resgate. Embora o passageiro tenha pago uma tarifa alta e embarcado no porto Cherbourg, esses fatores tiveram influências relativamente pequenas na predição. A combinação destes fatores principais, mulheres em alta classe social, foi o que determinou a so

In [6]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Iniciando a síntese com a API do Gemini...\n")

resposta_gemini = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_sintese
)

print("=== EXPLICAÇÃO FORTE (GEMINI LLM) ===")
print(resposta_gemini.text)

Iniciando a síntese com a API do Gemini...

=== EXPLICAÇÃO FORTE (GEMINI LLM) ===
O passageiro sobreviveu. Sua sobrevivência foi influenciada principalmente pelo fato de ser mulher, o que teve um impacto positivo e significativo na predição. Além disso, a vantagem de viajar na primeira classe também teve um impacto positivo, embora menos significativo. Outros fatores mencionados nas análises, como idade, número de parentes a bordo, tarifa paga e porto de embarque, apresentaram impactos mínimos ou contraditórios, não sendo considerados determinantes para o desfecho.
